# Batch MAELIA — Plan d'Expérience SMT sur terrainSA

Ce notebook lance un plan d'expérience SMT sur `terrainSA`, où la parcelle `beauce_5_1`
est clonée 100 fois sous le même îlot `beauce_5`. Chaque clone reçoit un point DOE
différent, ce qui amortit l'initialisation GAMA sur plusieurs points de sensibilité.

**Suivi des logs en temps réel** — ouvrir un terminal et lancer :
```bash
tail -f /tmp/gama_smt_terrainSA.log
```


Lorsque `WRITE_DYNAMIC_OUTPUTS = True`, GAMA écrit aussi `sorties_dynamiques_SA.csv`, une sortie journalière légère destinée aux PDP/ICE temporels (`N_lixi`, `DeltaCorg`, `Yield`).


## 0. Imports & configuration

In [ ]:
import os, sys, subprocess, time, io, warnings, shutil, re
from pathlib import Path
from datetime import date, timedelta
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# SMT
from smt.design_space import FloatVariable, OrdinalVariable, CategoricalVariable
from smt_design_space_ext import AdsgDesignSpaceImpl

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120, 'savefig.bbox': 'tight'})

# ─── Chemins ──────────────────────────────────────────────────────────────────
GAMA_HEADLESS = '/Applications/Gama.app/Contents/headless/gama-headless.sh'
def find_project_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'requirements.txt').exists() and (candidate / 'maelia_sa_pipeline').exists():
            return candidate
    raise RuntimeError('Racine du dépôt introuvable depuis le répertoire courant.')

PROJECT_ROOT  = find_project_root()
SIMULATIONS_DIR = PROJECT_ROOT / 'simulations'
DEFAULT_MAELIA_ROOT = Path.home() / 'Workspace_GAMA' / 'MAELIA'
MAELIA_ROOT   = Path(os.environ.get('MAELIA_ROOT', str(DEFAULT_MAELIA_ROOT))).expanduser()
MODEL_PATH    = MAELIA_ROOT / 'models/main/launcherBase.gaml'

def read_launcher_parameters(model_path):
    """Retourne les variables exposées comme paramètres headless par le launcher courant."""
    text = Path(model_path).read_text(errors='replace')
    params = {}
    for line in text.splitlines():
        stripped = line.strip()
        if not stripped or stripped.startswith('//'):
            continue
        match = re.search(r"parameter\s+(['\"])(.*?)\1\s+var:\s*([A-Za-z0-9_]+)", line)
        if match:
            params[match.group(3)] = match.group(2)
    return params

LAUNCHER_PARAMETERS = read_launcher_parameters(MODEL_PATH)

# GAMA reste intact : le terrainSA est généré côté projet puis transmis à GAMA
# via le paramètre cheminModeleVersDonnees.
if str(SIMULATIONS_DIR) not in sys.path:
    sys.path.insert(0, str(SIMULATIONS_DIR))
from build_terrainSA_project import ensure_terrain_sa, validate_terrain_sa

PROJECT_INCLUDES_ROOT = SIMULATIONS_DIR / 'gama_includes'
NOM_DECOUPAGE = 'terrainSA'
FORCE_REBUILD_TERRAINSA = True
TERRAINSA_DIR = ensure_terrain_sa(
    maelia_root=MAELIA_ROOT,
    project_includes_root=PROJECT_INCLUDES_ROOT,
    force=FORCE_REBUILD_TERRAINSA,
    n_clones=100,
)
TERRAINSA_INFO = validate_terrain_sa(TERRAINSA_DIR, n_clones=100)
CHEMIN_MODELE_VERS_DONNEES = str(PROJECT_INCLUDES_ROOT) + '/'

# MAELIA reconstruit son propre dossier de sortie dans models/main/log.
# Le notebook valide donc les logs a cet emplacement, puis copie les runs OK
# vers PROJECT_LOG_DIR pour les notebooks d'analyse et l'application web.
GAMA_OUTPUT_DIR = MAELIA_ROOT / 'models/main/log'
PROJECT_LOG_DIR = SIMULATIONS_DIR / 'log_terrainSA'
LOG_DIR       = GAMA_OUTPUT_DIR
VARIANTS_DIR  = TERRAINSA_DIR / 'modeleAgricole/agriculteurs/variants_SMT'
BLOC_DONNEES  = TERRAINSA_DIR / 'modeleAgricole/blocsDonnees.csv'
XML_DIR       = Path('/tmp/maelia_smt_terrainSA_xml')
GAMA_LOG      = Path('/tmp/gama_smt_terrainSA.log')

# Active une sortie journaliere supplementaire pour analyser les trajectoires
# avec PDP/ICE si le launcher courant expose cette option. Le plan d'experience
# SMT reste strictement identique.
REQUEST_DYNAMIC_OUTPUTS = True
DYNAMIC_OUTPUT_GLOB = 'sorties_dynamiques_SA*.csv'
WRITE_DYNAMIC_OUTPUTS = REQUEST_DYNAMIC_OUTPUTS and ('sorties_dynamiques_SA' in LAUNCHER_PARAMETERS)

# ─── Paramètres de simulation ─────────────────────────────────────────────────
ANNEE_DEBUT   = 2019
NB_ANNEES     = 12
ANNEES_DOSE   = list(range(2019, 2031))
FINAL_STEP    = 1200
CROP          = 'ble'
CROP_MAELIA   = 'BTH'
CHEMIN_RACINE = str(MAELIA_ROOT) + '/'

# ─── Plan d'expérience groupé ─────────────────────────────────────────────────
TARGET_N_DOE = 10000
CLONES_PER_RUN = 100

_all_parcs = []
with open(BLOC_DONNEES) as _f:
    for _line in _f:
        _parts = [_p.strip() for _p in _line.strip().split(';') if _p.strip()]
        if len(_parts) >= 2:
            _all_parcs.extend(_parts[1:])

ALL_PARCELLES = [p for p in _all_parcs if p.startswith('beauce_5_sa_')]
ALL_PARCELLES = sorted(ALL_PARCELLES)[:CLONES_PER_RUN]
if len(ALL_PARCELLES) != CLONES_PER_RUN:
    raise ValueError(f'terrainSA doit contenir {CLONES_PER_RUN} clones, trouvé {len(ALL_PARCELLES)}')

N_SIMU = int(np.ceil(TARGET_N_DOE / len(ALL_PARCELLES)))
N_DOE  = N_SIMU * len(ALL_PARCELLES)

# ─── Création des dossiers ────────────────────────────────────────────────────
XML_DIR.mkdir(parents=True, exist_ok=True)
VARIANTS_DIR.mkdir(parents=True, exist_ok=True)
PROJECT_LOG_DIR.mkdir(parents=True, exist_ok=True)

# ─── Nettoyage des anciens fichiers SMT terrainSA ─────────────────────────────
for _label, _root in [('GAMA', LOG_DIR), ('projet', PROJECT_LOG_DIR)]:
    old_dirs = sorted(_root.glob(f'{NOM_DECOUPAGE}_smt_*'))
    if old_dirs:
        print(f'Nettoyage de {len(old_dirs)} ancien(s) répertoire(s) SMT terrainSA ({_label})...')
        for d in old_dirs:
            shutil.rmtree(d)
        print('  OK.')

old_csvs = sorted(VARIANTS_DIR.glob('dateDose_smt_*.csv'))
if old_csvs:
    print(f'Nettoyage de {len(old_csvs)} anciens fichier(s) dateDose SMT terrainSA...')
    for f in old_csvs:
        f.unlink()
    print('  OK.')

print('Configuration OK')
print(f'  Terrain       : {NOM_DECOUPAGE}')
print(f'  Culture       : {CROP}')
print(f'  Campagnes     : {ANNEES_DOSE[0]}–{ANNEES_DOSE[-1]}')
print(f'  Simulations   : {N_SIMU} runs')
print(f'  Clones/run    : {len(ALL_PARCELLES)}')
print(f'  Points DOE    : {N_DOE}')
print(f'  Sorties GAMA  : {LOG_DIR}')
print(f'  Includes root : {PROJECT_INCLUDES_ROOT}')
print(f'  TerrainSA dir : {TERRAINSA_DIR}')
print(f'  Sol / ZH      : {TERRAINSA_INFO["soil"]} / {TERRAINSA_INFO["zh"]}')
print(f'  Sorties projet: {PROJECT_LOG_DIR}')
print(f'  Sorties dyn.  : {WRITE_DYNAMIC_OUTPUTS}')
if REQUEST_DYNAMIC_OUTPUTS and not WRITE_DYNAMIC_OUTPUTS:
    print('  Note          : sorties dynamiques désactivées, paramètre absent du launcher GAMA courant.')
print()
print('─' * 55)
print('Pour suivre les logs GAMA dans un terminal :')
print(f'  tail -f {GAMA_LOG}')
print('─' * 55)


## 1. Espace de conception SMT

Reprise de l'espace de conception hiérarchique courant : 15 variables, avec profondeurs explicites et fertilisant minéral fixé.

In [ ]:
# ─── Paramètres explorés : profondeurs explicites et engrais minéral unique ────
# Les anciens types de préparation étaient des alias de profondeur et ne sont donc
# plus explorés comme catégories. Les fertilisants sont fixés à un minéral unique
# représentatif du terrain réel : AN.
FERTILIZER_TYPE = 'AN'
PREPA_TYPE_1 = 'travail_sol_1'
PREPA_TYPE_2 = 'une_reprise'
SEMIS_TYPE = 'semis'
PREPA_DEPTH_MIN_CM = 1.0
PREPA_DEPTH_MAX_CM = 25.0
SEMIS_DEPTH_MIN_CM = 1.0
SEMIS_DEPTH_MAX_CM = 4.0

# Plan SMT/ADSG utilisé pour les simulations MAELIA.
# Les variables calendaires sont exprimées en jours de campagne : 1 = 1er août.
SMT_FEATURES = [
    'n_ferti', 'has_prepa', 'nb_prepa',
    'Date_Semis', 'Delta_PREPA_Semis', 'Profondeur_Semis',
    'Profondeur_Prepa_1', 'Profondeur_Prepa_2',
    'Date_F1', 'Date_F2', 'Date_F3', 'Date_Recolte',
    'Dose_F1', 'Dose_F2', 'Dose_F3',
]
SMT_CATEGORICAL = ['n_ferti', 'has_prepa', 'nb_prepa']
SMT_CONTINUOUS = [feature for feature in SMT_FEATURES if feature not in SMT_CATEGORICAL]

agri_design_space = AdsgDesignSpaceImpl(
    design_variables=[
        OrdinalVariable(['0_ferti', '1_ferti', '2_ferti', '3_ferti']),  # 0: n_ferti
        OrdinalVariable(['Non_prepa', 'Oui_prepa']),                    # 1: has_prepa
        OrdinalVariable(['1_prepa', '2_prepa']),                        # 2: nb_prepa

        FloatVariable(45, 106),                 # 3: Date_Semis
        FloatVariable(-44, -4),                 # 4: Delta_PREPA_Semis
        FloatVariable(SEMIS_DEPTH_MIN_CM, SEMIS_DEPTH_MAX_CM),          # 5: Profondeur_Semis
        FloatVariable(PREPA_DEPTH_MIN_CM, PREPA_DEPTH_MAX_CM),          # 6: Profondeur_Prepa_1
        FloatVariable(PREPA_DEPTH_MIN_CM, PREPA_DEPTH_MAX_CM),          # 7: Profondeur_Prepa_2
        FloatVariable(106, 323),                # 8: Date_F1
        FloatVariable(210, 323),                # 9: Date_F2
        FloatVariable(240, 323),                # 10: Date_F3
        FloatVariable(323, 384),                # 11: Date_Recolte
        FloatVariable(10, 100),                 # 12: Dose_F1
        FloatVariable(10, 100),                 # 13: Dose_F2
        FloatVariable(10, 100),                 # 14: Dose_F3
    ]
)

# Hiérarchie d'activation ADSG.
agri_design_space.declare_decreed_var(decreed_var=2, meta_var=1, meta_value=['Oui_prepa'])
agri_design_space.declare_decreed_var(decreed_var=4, meta_var=1, meta_value=['Oui_prepa'])
agri_design_space.declare_decreed_var(decreed_var=6, meta_var=1, meta_value=['Oui_prepa'])
agri_design_space.declare_decreed_var(decreed_var=7, meta_var=2, meta_value=['2_prepa'])

agri_design_space.declare_decreed_var(decreed_var=8,  meta_var=0, meta_value=['1_ferti', '2_ferti', '3_ferti'])
agri_design_space.declare_decreed_var(decreed_var=12, meta_var=0, meta_value=['1_ferti', '2_ferti', '3_ferti'])
agri_design_space.declare_decreed_var(decreed_var=9,  meta_var=0, meta_value=['2_ferti', '3_ferti'])
agri_design_space.declare_decreed_var(decreed_var=13, meta_var=0, meta_value=['2_ferti', '3_ferti'])
agri_design_space.declare_decreed_var(decreed_var=10, meta_var=0, meta_value=['3_ferti'])
agri_design_space.declare_decreed_var(decreed_var=14, meta_var=0, meta_value=['3_ferti'])

DEFAULT_FLOATS = [
    75.0, -20.0, 3.0, 15.0, 10.0,
    150.0, 230.0, 280.0, 350.0,
    50.0, 50.0, 50.0,
]

def parse_smt_row(row_data):
    """Analyse une ligne décodée par SMT avec le plan terrainSA compact."""
    parsed = {
        'n_ferti': 0,
        'has_prepa': False,
        'nb_prepa': 1,
        'floats': [],
    }
    values = list(row_data.values()) if isinstance(row_data, dict) else row_data

    for val in values:
        if isinstance(val, (float, int, np.floating, np.integer)):
            parsed['floats'].append(float(val))
        elif isinstance(val, str):
            if val in ['0_ferti', '1_ferti', '2_ferti', '3_ferti']:
                parsed['n_ferti'] = int(val[0])
            elif val in ['Oui_prepa', 'Non_prepa']:
                parsed['has_prepa'] = (val == 'Oui_prepa')
            elif val in ['1_prepa', '2_prepa']:
                parsed['nb_prepa'] = int(val[0])
    return parsed


def normalized_floats(parsed):
    fl = list(parsed['floats'])
    for default in DEFAULT_FLOATS[len(fl):]:
        fl.append(default)
    return fl[:len(DEFAULT_FLOATS)]


def ordered_fertilisation_dates(parsed, fl):
    """Retourne des dates F1/F2/F3 strictement ordonnées pour MAELIA."""
    dates = {'f1': None, 'f2': None, 'f3': None}
    if parsed['n_ferti'] >= 1:
        dates['f1'] = int(round(fl[5]))
    if parsed['n_ferti'] >= 2:
        dates['f2'] = max(int(round(fl[6])), dates['f1'] + 1)
    if parsed['n_ferti'] >= 3:
        dates['f3'] = max(int(round(fl[7])), dates['f2'] + 1)
    return dates


def extract_table_from_smt(x_array, dsg_space):
    rows = []
    decoded_data = dsg_space.decode_values(x_array)

    for row_data in decoded_data:
        p = parse_smt_row(row_data)
        fl = normalized_floats(p)
        ferti_dates = ordered_fertilisation_dates(p, fl)

        semi = int(round(fl[0]))
        prepa_date = int(round(semi + fl[1])) if p['has_prepa'] else None

        rows.append({
            'n_ferti': p['n_ferti'],
            'has_prepa': 'Oui' if p['has_prepa'] else 'Non',
            'nb_prepa': p['nb_prepa'] if p['has_prepa'] else '-',
            'Semi': semi,
            'Delta_PREPA_Semis': round(fl[1], 1) if p['has_prepa'] else '-',
            'Profondeur_Semis': round(fl[2], 1),
            'PREPA_Date': prepa_date if prepa_date is not None else '-',
            'Profondeur_Prepa_1': round(fl[3], 1) if p['has_prepa'] else '-',
            'Profondeur_Prepa_2': round(fl[4], 1) if p['has_prepa'] and p['nb_prepa'] == 2 else '-',
            'Ferti_1': ferti_dates['f1'] if ferti_dates['f1'] is not None else '-',
            'Dose_F1': round(fl[9], 1) if p['n_ferti'] >= 1 else '-',
            'Ferti_2': ferti_dates['f2'] if ferti_dates['f2'] is not None else '-',
            'Dose_F2': round(fl[10], 1) if p['n_ferti'] >= 2 else '-',
            'Ferti_3': ferti_dates['f3'] if ferti_dates['f3'] is not None else '-',
            'Dose_F3': round(fl[11], 1) if p['n_ferti'] >= 3 else '-',
            'Recolte': int(round(fl[8])),
            'Engrais': FERTILIZER_TYPE,
        })

    return pd.DataFrame(rows)


## 2. Génération du plan d'expérience (DOE)

In [ ]:
print(f'Génération de {N_DOE} points LHS hiérarchiques (SMT ADSG)...')
result = agri_design_space._sample_valid_x(N_DOE)
xt, is_acting_t = result[0], result[1]
print(f'Matrice DOE : {xt.shape[0]} × {xt.shape[1]} variables')
print()

doe_display = extract_table_from_smt(xt, agri_design_space)
print(f'Aperçu du plan d\'expérience (N={N_DOE}) :')
display(doe_display)

## 3. Génération des fichiers `dateDose.csv`

Chaque point du DOE est converti en fichier `dateDose_smt_XXX.csv` au format MAELIA :
- **PREPA** : `fl[1]` jours **avant** le semis (préparation du sol).
- **SEMIS** : jour `fl[0]` de l'année.
- **FERTI_Min** : `fl[2]` jours après semis (F1), puis `fl[3]` jours pour F2, `fl[4]` pour F3.
- **RECOLTE** : `fl[5]` jours après la dernière opération.

In [ ]:
COLS = ['Trait', 'nom_MAELIA', 'plant_sem', 'annee_deb',
        'id_operation', 'ordre_op', 'DATE', 'PROF', 'TYPE', 'DOSE', 'TEMPS']

HARVEST_BUFFER_DAYS = 7
MIN_DAYS_LAST_OP_TO_HARVEST = 1

def days_in_year(year):
    # Longueur de la campagne agricole démarrant le 1er août de `year`.
    return (date(year + 1, 8, 1) - date(year, 8, 1)).days

def doy_to_date_str(year, doy):
    # Les jours du plan SMT sont des jours de campagne : 1 = 1er août de `year`.
    campaign_start = date(year, 8, 1)
    return (campaign_start + timedelta(days=int(doy) - 1)).strftime('%d/%m/%Y')

def parse_date_str(value):
    return pd.to_datetime(value, dayfirst=True).date()

def safe_f(val, default=50.0):
    try:
        v = float(val)
        return default if v != v else v
    except (TypeError, ValueError):
        return default

def validate_datedose_rows(rows):
    """Détecte les calendriers qui feraient planter agriculteurDateDose.gaml."""
    order = {'RECOLTE': 0, 'PREPA': 1, 'SEMIS': 2, 'FERTI_Min': 3}
    by_parcelle = {}
    issues = []
    for row in rows:
        by_parcelle.setdefault(row['Trait'], []).append(row)

    for parcelle_id, parcelle_ops in by_parcelle.items():
        culture_en_place = False
        sorted_ops = sorted(
            parcelle_ops,
            key=lambda row: (parse_date_str(row['DATE']),
                             order.get(row['id_operation'], 99),
                             str(row.get('ordre_op', ''))),
        )
        for row in sorted_ops:
            op = row['id_operation']
            day = row['DATE']
            campagne = int(row['annee_deb'])
            op_date = parse_date_str(day)
            campaign_start = date(campagne, 8, 1)
            if op_date < campaign_start:
                issues.append(
                    f'{parcelle_id} campagne {campagne}: {op} le {day} avant le démarrage de campagne ({campaign_start.strftime("%d/%m/%Y")})'
                )
            if op == 'SEMIS':
                if culture_en_place:
                    issues.append(f'{parcelle_id} campagne {campagne}: SEMIS le {day} alors qu’une culture est déjà en place')
                culture_en_place = True
            elif op == 'RECOLTE':
                if not culture_en_place:
                    issues.append(f'{parcelle_id} campagne {campagne}: RECOLTE le {day} sans culture en place')
                culture_en_place = False
            elif op == 'FERTI_Min':
                if not culture_en_place:
                    issues.append(f'{parcelle_id} campagne {campagne}: FERTI_Min le {day} sans culture en place')
            elif op == 'PREPA':
                pass
            else:
                issues.append(f'{parcelle_id} campagne {campagne}: opération inconnue {op}')
    return issues

def generate_ops_for_parcelle(point_idx, parcelle_id, years):
    """Génère les opérations pour un SEUL point DOE appliqué à une SEULE parcelle."""
    decoded = agri_design_space.decode_values(xt[point_idx:point_idx + 1])
    p = parse_smt_row(decoded[0])
    fl = [safe_f(v, default=d) for v, d in zip(normalized_floats(p), DEFAULT_FLOATS)]

    ops = []
    for year in years:
        semi_doy = max(1, int(round(fl[0])))
        prepa_doy = max(1, int(round(semi_doy + fl[1]))) if p['has_prepa'] else None
        ferti_dates = ordered_fertilisation_dates(p, fl)

        next_entry_doy = prepa_doy if prepa_doy is not None else semi_doy
        latest_rec_doy = days_in_year(year) + next_entry_doy - HARVEST_BUFFER_DAYS
        max_last_op_doy = latest_rec_doy - MIN_DAYS_LAST_OP_TO_HARVEST

        ferti_events = []
        if ferti_dates['f1'] is not None:
            ferti_events.append((ferti_dates['f1'], fl[9]))
        if ferti_dates['f2'] is not None:
            ferti_events.append((ferti_dates['f2'], fl[10]))
        if ferti_dates['f3'] is not None:
            ferti_events.append((ferti_dates['f3'], fl[11]))
        ferti_events = [event for event in ferti_events if event[0] <= max_last_op_doy]

        last_doy = ferti_events[-1][0] if ferti_events else semi_doy
        rec_doy = max(int(round(fl[8])), last_doy + MIN_DAYS_LAST_OP_TO_HARVEST)
        rec_doy = min(rec_doy, latest_rec_doy)
        if rec_doy <= last_doy:
            raise ValueError(
                f'Calendrier impossible pour {parcelle_id} campagne {year}: '
                f'dernière opération DOY {last_doy}, récolte max DOY {latest_rec_doy}'
            )

        base = {'Trait': parcelle_id, 'nom_MAELIA': CROP_MAELIA,
                'plant_sem': CROP, 'annee_deb': year}

        if p['has_prepa']:
            ops.append({**base, 'id_operation': 'PREPA', 'ordre_op': '',
                        'DATE': doy_to_date_str(year, prepa_doy),
                        'PROF': str(round(fl[3], 1)), 'TYPE': PREPA_TYPE_1, 'DOSE': ''})
            if p['nb_prepa'] == 2:
                ops.append({**base, 'id_operation': 'PREPA', 'ordre_op': '2',
                            'DATE': doy_to_date_str(year, prepa_doy),
                            'PROF': str(round(fl[4], 1)), 'TYPE': PREPA_TYPE_2, 'DOSE': ''})

        ops.append({**base, 'id_operation': 'SEMIS', 'ordre_op': '',
                    'DATE': doy_to_date_str(year, semi_doy),
                    'PROF': str(round(fl[2], 1)), 'TYPE': SEMIS_TYPE, 'DOSE': ''})

        for ordre, (doy, dose) in enumerate(ferti_events, 1):
            ops.append({**base, 'id_operation': 'FERTI_Min', 'ordre_op': str(ordre),
                        'DATE': doy_to_date_str(year, doy), 'PROF': '',
                        'TYPE': FERTILIZER_TYPE, 'DOSE': str(round(dose, 1))})

        ops.append({**base, 'id_operation': 'RECOLTE', 'ordre_op': '',
                    'DATE': doy_to_date_str(year, rec_doy),
                    'PROF': '', 'TYPE': '', 'DOSE': ''})

    for op in ops:
        op.setdefault('TEMPS', '')
    return ops

# ─── Générer tous les fichiers dateDose ───────────────────────────────────────
print(f'Génération de {N_SIMU} fichiers dateDose distribués...')
for i in range(N_SIMU):
    path = VARIANTS_DIR / f'dateDose_smt_{i:03d}.csv'
    all_ops = []
    for j, parcelle_id in enumerate(ALL_PARCELLES):
        point_idx = i * len(ALL_PARCELLES) + j
        all_ops.extend(generate_ops_for_parcelle(point_idx, parcelle_id, ANNEES_DOSE))

    issues = validate_datedose_rows(all_ops)
    if issues:
        preview = '\n  - '.join(issues[:10])
        raise ValueError(f'Calendrier dateDose invalide pour simulation {i:03d}:\n  - {preview}')

    df = pd.DataFrame(all_ops, columns=COLS)
    df.to_csv(path, sep=';', index=False)
    if i % 10 == 0 or i == N_SIMU - 1:
        print(f'  Simulation {i:03d} : {len(df)} opérations ({len(ALL_PARCELLES)} parcelles traitées)')

# Contrôle anti-régression : si les dates ont été converties depuis le 1er janvier
# au lieu du 1er août, MAELIA démarre après les semis et les logs restent vides.
_check = pd.read_csv(VARIANTS_DIR / 'dateDose_smt_000.csv', sep=';')
_check_semis = pd.to_datetime(
    _check.loc[_check['id_operation'] == 'SEMIS', 'DATE'],
    dayfirst=True,
    errors='coerce',
)
_check_campaign = pd.to_datetime(
    _check.loc[_check['id_operation'] == 'SEMIS', 'annee_deb'].astype(int).astype(str) + '-08-01',
    errors='coerce',
)
if (_check_semis.reset_index(drop=True) < _check_campaign.reset_index(drop=True)).any():
    _bad = _check.loc[_check['id_operation'] == 'SEMIS'].head(5)
    raise ValueError(
        'Dates de semis invalides: au moins un semis est avant le 1er août de sa campagne. '
        'Recharge le notebook depuis le fichier modifié, puis relance les cellules de définition du plan et de génération dateDose.\n'
        + _bad.to_string(index=False)
    )

_depth_check = _check[_check['id_operation'].isin(['SEMIS', 'PREPA'])].copy()
_depth_check['PROF_num'] = pd.to_numeric(_depth_check['PROF'], errors='coerce')
if _depth_check['PROF_num'].isna().any() or (_depth_check['PROF_num'] <= 0).any():
    raise ValueError('Les opérations SEMIS/PREPA doivent porter une profondeur strictement positive.\n' + _depth_check.head(10).to_string(index=False))

print(f'\nFichiers écrits dans : {VARIANTS_DIR}')
print(f'Engrais minéral fixé pour tous les apports : {FERTILIZER_TYPE}')


In [ ]:
df.head(10)

## 4. Lancement des simulations GAMA (headless)

Chaque simulation :
1. Génère un fichier XML de plan d'expérience GAMA.
2. Lance `gama-headless.sh` avec ce XML.
3. Écrit les logs dans `/tmp/gama_smt.log` sans les streamer dans la cellule.

> Pour suivre les logs depuis un terminal : `tail -f /tmp/gama_smt.log`


Si `WRITE_DYNAMIC_OUTPUTS=True`, chaque run produit en plus un fichier `sorties_dynamiques_SA.csv` avec une ligne par jour et par clone terrainSA.


In [ ]:
# ─── Générateur XML headless ──────────────────────────────────────────────────

def make_xml(nom_simu, datedose_path):
    """Génère le contenu XML du plan d'expérience GAMA headless."""
    datedose_path = Path(datedose_path)
    try:
        # MAELIA préfixe fichierDateDose par le répertoire du terrain courant.
        # Il faut donc fournir un chemin interne à terrainSA, pas un chemin absolu.
        datedose_for_gama = '/' + datedose_path.relative_to(TERRAINSA_DIR).as_posix()
    except ValueError:
        datedose_for_gama = datedose_path.as_posix()

    root = ET.Element('Experiment_plan')
    sim  = ET.SubElement(root, 'Simulation')
    sim.set('experiment',  'simulationBase')
    sim.set('finalStep',   str(FINAL_STEP))
    sim.set('id',          '0')
    sim.set('seed',        '1.0')
    sim.set('sourcePath',  str(MODEL_PATH))

    params = ET.SubElement(sim, 'Parameters')

    skipped_params = []

    def p(name, ptype, value, var):
        if var not in LAUNCHER_PARAMETERS:
            skipped_params.append(var)
            return
        el = ET.SubElement(params, 'Parameter')
        el.set('name', name)
        el.set('type', ptype)
        el.set('value', str(value))
        el.set('var', var)

    p('executerSurCluster: ',                  'BOOLEAN', 'false',                               'executerSurCluster')
    p('cheminRacineMaelia',                    'STRING',  CHEMIN_RACINE,                         'cheminRacineMaelia')
    p('cheminModeleVersDonnees',               'STRING',  CHEMIN_MODELE_VERS_DONNEES,             'cheminModeleVersDonnees')
    p('cheminSorties',                         'STRING',  str(LOG_DIR),                          'cheminRelatifDuDossierDeSortieDeSimulation')
    p('anneeDebutSimulation : ',               'INT',     ANNEE_DEBUT,                           'anneeDebutSimulation')
    p('nbAnneesSimulation : ',                 'INT',     NB_ANNEES,                             'nbAnneesSimulation')
    p('nomSimulation : ',                      'STRING',  nom_simu,                              'nomSimulation')
    p('nomDecoupageZonePourLectureFichiers : ', 'STRING',  NOM_DECOUPAGE,                         'nomDecoupageZonePourLectureFichiers')
    p('modeVerbeux :',                         'BOOLEAN', 'false',                               'verboseMode')
    p('simulationSurParcelle : ',              'BOOLEAN', 'false',                               'executerUneSeuleParcelle')
    p('idParcelleASimuler : ',                 'STRING',  ALL_PARCELLES[0],                      'nomParcelleAffichee')
    p('executerModeleAgricole : ',             'BOOLEAN', 'true',                                'executerModeleAgricole')
    p('nomChoixAssolement : ',                 'STRING',  'DateDose',                            'nomChoixAssolement')
    p('fichierDateDose : ',                    'STRING',  datedose_for_gama,                     'fichierDateDose')
    p('avecContrainteDeMainOeuvre : ',         'BOOLEAN', 'false',                               'avecContrainteDeMainOeuvre')
    p('isIrrigationSimulee : ',                'BOOLEAN', 'false',                               'isIrrigationSimulee')
    p('executerModeleHydrographique : ',       'BOOLEAN', 'false',                               'executerModeleHydrographique')
    p('executerModeleNormatif : ',             'BOOLEAN', 'false',                               'executerModeleNormatif')
    p('executerEcritureFichiers : ',           'BOOLEAN', 'true',                                'executerEcritureFichiers')
    p('sorties eau',                           'BOOLEAN', 'true',                                'sorties_eau')
    p('sorties azote',                         'BOOLEAN', 'true',                                'sorties_azote')
    p('sorties carbone et GES',                'BOOLEAN', 'true',                                'sorties_carboneGES')
    if WRITE_DYNAMIC_OUTPUTS:
        p('Sorties dynamiques SA',              'BOOLEAN', 'true',                                'sorties_dynamiques_SA')
    p('Sortie N_Cstock_Parcelles',             'BOOLEAN', 'true',                                'N_Cstock_Parcelles')
    p('Sortie N_lixi_Parcelles',               'BOOLEAN', 'true',                                'N_lixi_Parcelles')
    # Note: 'suiviOTParParcelle' est déjà true par défaut dans launcherBase.gaml.
    # Son label GAML contient des accents ('Suivi détaillé…') qui ne peuvent pas
    # être encodés sans risque dans le XML headless → paramètre omis volontairement.

    if skipped_params and nom_simu == 'smt_000':
        skipped = ', '.join(sorted(set(skipped_params)))
        print(f'  Paramètres XML ignorés car absents du launcher courant : {skipped}', flush=True)

    ET.SubElement(sim, 'Outputs')

    try:
        ET.indent(root, space='  ')
    except AttributeError:
        pass  # Python < 3.9

    buf = io.BytesIO()
    ET.ElementTree(root).write(buf, encoding='UTF-8', xml_declaration=True)
    xml_str = buf.getvalue().decode('UTF-8')
    # Normaliser la déclaration XML (double quotes)
    xml_str = xml_str.replace("<?xml version='1.0' encoding='UTF-8'?>",
                              '<?xml version="1.0" encoding="UTF-8" standalone="no"?>')
    return xml_str


# ─── Fonction de lancement + validation des logs ──────────────────────────────

def find_output_dir(nom_simu, ts_before):
    """Trouve le répertoire de sortie MAELIA créé après ts_before."""
    expected_prefix = f'{NOM_DECOUPAGE}_{nom_simu}_'
    for _ in range(20):
        matches = [
            p for p in LOG_DIR.glob(f'{NOM_DECOUPAGE}_*')
            if p.is_dir()
            and p.stat().st_mtime >= ts_before
            and (p.name.startswith(expected_prefix) or nom_simu in p.name)
        ]
        if matches:
            return sorted(matches, key=lambda p: p.stat().st_mtime)[-1]
        time.sleep(1)
    return None


def console_text(gama_ws):
    console = gama_ws / 'console-outputs-0.txt'
    if console.exists():
        return console.read_text(errors='replace')
    return ''


def has_gama_failure(text):
    markers = [
        "The model couldn't be compiled",
        "Model didn't compile",
        "ERREUR LORS DE L'INITIALISATION",
        "ERREUR LORS DE LA SIMULATION",
        "Fichier date/dose inexistant",
    ]
    return any(marker in text for marker in markers)


def output_dir_has_required_logs(path):
    """Valide qu'un run a vraiment écrit les sorties nécessaires à l'analyse."""
    if path is None or not path.exists():
        return False
    base_files = ['sorties_CN.csv', 'sorties_GES.csv', 'suiviOTParParcelle.csv']
    has_base = all((path / name).exists() for name in base_files)
    has_dynamic = (not WRITE_DYNAMIC_OUTPUTS) or any(path.glob(DYNAMIC_OUTPUT_GLOB))
    return has_base and has_dynamic


def missing_required_logs(path):
    if path is None or not path.exists():
        return ['dossier de sortie introuvable']
    missing = [name for name in ['sorties_CN.csv', 'sorties_GES.csv', 'suiviOTParParcelle.csv'] if not (path / name).exists()]
    if WRITE_DYNAMIC_OUTPUTS and not any(path.glob(DYNAMIC_OUTPUT_GLOB)):
        missing.append(DYNAMIC_OUTPUT_GLOB)
    return missing


def mirror_output_dir(out_dir):
    """Copie un run validé vers le répertoire lu par les analyses et l'app web."""
    PROJECT_LOG_DIR.mkdir(parents=True, exist_ok=True)
    dest = PROJECT_LOG_DIR / out_dir.name
    if dest.exists():
        shutil.rmtree(dest)
    shutil.copytree(out_dir, dest)
    return dest


def run_simulation(nom_simu, xml_path, gama_ws):
    """
    Lance une simulation GAMA headless.
    Écrit stdout/stderr dans GAMA_LOG sans streamer dans la cellule notebook.
    Retourne (code_retour, out_dir | None), avec out_dir dans PROJECT_LOG_DIR.
    """
    gama_ws.mkdir(parents=True, exist_ok=True)
    cmd = ['bash', GAMA_HEADLESS, str(xml_path), str(gama_ws)]
    t0  = time.time()
    ts_before = t0 - 0.5

    print(f'\n{nom_simu} lancé. Logs terminal : tail -f {GAMA_LOG}', flush=True)

    with open(GAMA_LOG, 'a') as flog:
        flog.write(f'\n{"=" * 62}\n[{nom_simu}] {time.strftime("%H:%M:%S")}\n')
        flog.write(f'Commande: {" ".join(cmd)}\n{"=" * 62}\n')
        proc = subprocess.run(
            cmd,
            stdout=flog,
            stderr=subprocess.STDOUT,
            text=True,
        )

    elapsed = time.time() - t0
    out_dir = find_output_dir(nom_simu, ts_before) if proc.returncode == 0 else None
    ctext = console_text(gama_ws)
    missing = missing_required_logs(out_dir)
    ok = (proc.returncode == 0) and (not has_gama_failure(ctext)) and (not missing)
    project_out_dir = mirror_output_dir(out_dir) if ok else None
    status = 'OK' if ok else f'ERREUR (code {proc.returncode})'

    print(f'  [{status}] {nom_simu} terminé en {elapsed:.1f}s', flush=True)
    if project_out_dir:
        print(f'  Sorties GAMA   : {out_dir.name}', flush=True)
        print(f'  Sorties projet : {project_out_dir.name}', flush=True)
    else:
        print(f'  Sorties invalides : {", ".join(missing)}', flush=True)
        if has_gama_failure(ctext):
            print('  La console GAMA signale une erreur de compilation ou d\'initialisation.', flush=True)
    return proc.returncode, project_out_dir


In [ ]:
# ─── Boucle principale ────────────────────────────────────────────────────────

expected_datedose = [VARIANTS_DIR / f'dateDose_smt_{i:03d}.csv' for i in range(N_SIMU)]
missing_datedose = [p for p in expected_datedose if not p.exists()]
if missing_datedose:
    preview = '\n  - '.join(str(p) for p in missing_datedose[:10])
    raise FileNotFoundError(
        f'{len(missing_datedose)}/{N_SIMU} fichier(s) dateDose SMT manquant(s). '
        'Relance d’abord la section 3 "Génération des fichiers dateDose.csv".\n'
        f'  - {preview}'
    )

repaired_datedose = 0
for _path in expected_datedose:
    _df = pd.read_csv(_path, sep=';', dtype=str).fillna('')
    if 'TEMPS' not in _df.columns:
        _df['TEMPS'] = ''
        _df.to_csv(_path, sep=';', index=False)
        repaired_datedose += 1
if repaired_datedose:
    print(f'Format DateDose mis à jour : colonne TEMPS ajoutée à {repaired_datedose} fichier(s).')

# Vider le fichier de log
GAMA_LOG.write_text('')
print(f'Log GAMA → {GAMA_LOG}')
print('Pour suivre en temps réel dans un terminal :')
print(f'  tail -f {GAMA_LOG}')
print()

results = {}    # nom_simu → Path | None, dans PROJECT_LOG_DIR
t_global = time.time()

for i in range(N_SIMU):
    nom_simu  = f'smt_{i:03d}'
    csv_path  = VARIANTS_DIR / f'dateDose_{nom_simu}.csv'
    xml_path  = XML_DIR / f'{nom_simu}.xml'
    gama_ws   = XML_DIR / f'ws_{nom_simu}'

    # Écrire le XML
    xml_path.write_text(make_xml(nom_simu, csv_path))

    # Lancer la simulation
    ret, out_dir = run_simulation(nom_simu, xml_path, gama_ws)
    results[nom_simu] = out_dir

n_ok    = sum(1 for v in results.values() if v is not None)
elapsed = time.time() - t_global
print(f'\n{"═" * 62}')
print(f'  Terminé : {n_ok}/{N_SIMU} simulations OK  ({elapsed:.0f}s total)')
print(f'{"═" * 62}')
if n_ok != N_SIMU:
    print('Au moins une simulation n’a pas produit tous les logs attendus : consulte le diagnostic ci-dessous.')


In [ ]:
# ─── Reconstruction des résultats depuis le répertoire de logs ────────────────
# Exécuter cette cellule si les simulations ont déjà tourné mais que `results`
# n'est plus en mémoire (kernel redémarré, interruption, etc.).

def _output_dir_has_expected_files(path):
    return output_dir_has_required_logs(path)

def _batch_start_cutoff():
    times = []
    for i in range(N_SIMU):
        console = XML_DIR / f'ws_smt_{i:03d}' / 'console-outputs-0.txt'
        if console.exists():
            times.append(console.stat().st_mtime)
    return min(times) - 120 if times else 0

def rebuild_results_from_logs():
    latest = {}
    unmatched = []
    cutoff = _batch_start_cutoff()

    for d in PROJECT_LOG_DIR.glob(f'{NOM_DECOUPAGE}_*'):
        if not d.is_dir() or not _output_dir_has_expected_files(d):
            continue
        if 'smoke_patch' in d.name or 'smoke_dyn' in d.name or d.stat().st_mtime < cutoff:
            continue

        matched = False
        for i in range(N_SIMU):
            nom_simu = f'smt_{i:03d}'
            if f'_{nom_simu}_' in d.name or d.name.startswith(f'{NOM_DECOUPAGE}_{nom_simu}_'):
                if nom_simu not in latest or d.stat().st_mtime > latest[nom_simu].stat().st_mtime:
                    latest[nom_simu] = d
                matched = True
                break

        if not matched:
            unmatched.append(d)

    # Compatibility with logs produced before the launcher fix, when all folders
    # could be named terrainTest_beauce_5_* instead of terrainTest_smt_XXX_*.
    missing = [f'smt_{i:03d}' for i in range(N_SIMU) if f'smt_{i:03d}' not in latest]
    for nom_simu, d in zip(missing, sorted(unmatched, key=lambda p: p.stat().st_mtime)):
        latest[nom_simu] = d

    return {k: v for k, v in sorted(latest.items())}

results = rebuild_results_from_logs()

n_found = len(results)
print(f'Dossiers trouvés : {n_found}/{N_SIMU}')
for nom, d in list(results.items())[:10]:
    print(f'  {nom} → {d.name}')
if n_found > 10:
    print(f'  … ({n_found - 10} autres)')
if n_found == 0:
    print(f'Aucun dossier exploitable trouvé sous {PROJECT_LOG_DIR}')


## 5. Collecte des résultats

In [ ]:
# ─── Diagnostic post-run ─────────────────────────────────────────────────────
print('═' * 62)
print('DIAGNOSTIC')
print('═' * 62)

for i in range(N_SIMU):
    nom_simu = f'smt_{i:03d}'
    gama_ws  = XML_DIR / f'ws_{nom_simu}'
    out_dir  = results.get(nom_simu)

    print(f'\n── {nom_simu} ──')

    # 1. Code retour GAMA
    console = gama_ws / 'console-outputs-0.txt'
    if console.exists():
        lines = console.read_text(errors='replace').splitlines()
        # Dernières lignes utiles (filtrer le bruit)
        tail = [l for l in lines if l.strip() and not l.startswith('Adding unit')][-10:]
        print('  console (10 dernières lignes utiles) :')
        for l in tail:
            print(f'    {l}')
    else:
        print(f'  console-outputs-0.txt introuvable dans {gama_ws}')

    # 2. Dossier de sortie MAELIA trouvé ?
    if out_dir:
        print(f'  → dossier sortie : {out_dir}')
        csvs = list(out_dir.glob('*.csv'))
        print(f'  → fichiers CSV   : {[c.name for c in csvs]}')
    else:
        print(f'  → aucun dossier sortie exploitable trouvé sous {PROJECT_LOG_DIR}')
        # Cherche quand même des dossiers récents pour aider au diagnostic
        all_dirs = sorted(LOG_DIR.glob(f'{NOM_DECOUPAGE}_*'), key=lambda p: p.stat().st_mtime, reverse=True)
        if all_dirs:
            print(f'  → dossiers GAMA les plus récents dans log/ :')
            for d in all_dirs[:5]:
                print(f'      {d.name}')
        else:
            print(f'  → log/ est vide ou n\'existe pas')


In [ ]:
import struct as _struct, glob as _glob

# --- Lecture DBF ilots (une seule fois) ---
def _read_dbf_ilot_sol(dbf_path):
    """Renvoie {ilot_name: ID_SOL} depuis le shapefile ilots.dbf."""
    with open(dbf_path, 'rb') as _f:
        _d = _f.read()
    _n   = _struct.unpack('<I', _d[4:8])[0]
    _hs  = _struct.unpack('<H', _d[8:10])[0]
    _rs  = _struct.unpack('<H', _d[10:12])[0]
    _flds = []
    _off = 32
    while _d[_off] != 0x0D:
        _name = _d[_off:_off+11].split(b'\x00')[0].decode('latin1')
        _flds.append((_name, _d[_off+16]))
        _off += 32
    _off = _hs
    _out = {}
    for _ in range(_n):
        if _d[_off] == 0x2A:
            _off += _rs
            continue
        _off += 1
        _row = {}
        for _nm, _ln in _flds:
            _row[_nm] = _d[_off:_off+_ln].decode('latin1').strip()
            _off += _ln
        _out[_row['ID_ILOT']] = _row['ID_SOL']
    return _out

_DBF_PATH = (PROJECT_INCLUDES_ROOT / NOM_DECOUPAGE / 'modeleAgricole/ilots/dansZone/ilots.dbf')
_ILOT_SOL = _read_dbf_ilot_sol(_DBF_PATH)   # {ilot_name -> sol_type}


def _read_corr_ilot_meteo(out_dir):
    """Renvoie {ilot_name: zone_meteo_int} depuis corresponsanceIlotZoneMeteo.csv."""
    _p = out_dir / 'corresponsanceIlotZoneMeteo.csv'
    if not _p.exists():
        return {}
    _df = pd.read_csv(_p, sep=';')
    return dict(zip(_df['ilot'], _df['zoneMeteo'].astype(int)))


SOL_TYPES   = ['18960_27491', '330104_330367_1', '330151_330489_1']  # A, B, C
METEO_CODES = [4756, 13218, 13761]  # sudouest, beauce, oceanique


def _zone(parcelle):
    for z in ('beauce', 'oceanique', 'sudouest'):
        if z in parcelle:
            return z
    return 'autre'


def read_simulation_outputs(out_dir):
    """
    Retourne un DataFrame long (parcelle x annee) avec les 3 sorties cibles.
    Colonnes : annee, parcelle, N_lixi, dCorg, rdt, zone, sol_type, zone_meteo
    """
    f_cn = out_dir / 'sorties_CN.csv'
    if not f_cn.exists():
        return pd.DataFrame()
    df = pd.read_csv(f_cn, sep=';',
                     usecols=['annee', 'parcelle', 'couvert',
                              'N_lixivie[kgN/ha]', 'delta_Corg[kgC/ha]'])
    df = df[df['couvert'].notna() & (df['couvert'] != 'solnu')].drop(columns='couvert')
    df = df.rename(columns={'N_lixivie[kgN/ha]': 'N_lixi',
                             'delta_Corg[kgC/ha]': 'dCorg'})

    f_ot = out_dir / 'suiviOTParParcelle.csv'
    if f_ot.exists():
        df_ot = pd.read_csv(f_ot, sep=';',
                            usecols=['annee', 'parcelle', 'OT',
                                     'RECOLTE_rendement[t/ha]'])
        df_ot = (df_ot[df_ot['OT'] == 'RECOLTE']
                 .drop(columns='OT')
                 .rename(columns={'RECOLTE_rendement[t/ha]': 'rdt'}))
        df = df.merge(df_ot, on=['annee', 'parcelle'], how='left')

    df['zone'] = df['parcelle'].apply(_zone)

    # sol_type et zone_meteo depuis shapefile + corresponsance
    _corr = _read_corr_ilot_meteo(out_dir)
    def _ilot_from_parcelle(p):
        if str(p).startswith('beauce_5_sa_'):
            return 'beauce_5'
        return '_'.join(str(p).rsplit('_', 1)[:-1])
    def _sol_type(p):
        return _ILOT_SOL.get(_ilot_from_parcelle(p), None)
    def _zone_meteo(p):
        return _corr.get(_ilot_from_parcelle(p), None)

    df['sol_type']   = df['parcelle'].apply(_sol_type)
    df['zone_meteo'] = df['parcelle'].apply(_zone_meteo)
    return df


# --- Collecte ---
all_data     = {}
summary_rows = []

for nom_simu, out_dir in results.items():
    if not (out_dir and out_dir.exists()):
        summary_rows.append({
            'simulation': nom_simu,
            'ok': False,
            'n_parcelles': 0,
            'reason': 'répertoire de sortie absent',
        })
        continue

    df = read_simulation_outputs(out_dir)
    if df.empty:
        summary_rows.append({
            'simulation': nom_simu,
            'ok': False,
            'n_parcelles': 0,
            'reason': 'sorties_CN.csv absent ou sans couvert cultivé',
        })
        continue

    all_data[nom_simu] = df
    row = {
        'simulation': nom_simu,
        'ok': True,
        'n_parcelles': df['parcelle'].nunique(),
        'reason': '',
    }
    for m in ('N_lixi', 'dCorg', 'rdt'):
        if m in df.columns:
            row[f'{m}_mean'] = df[m].mean()
            row[f'{m}_std']  = df[m].std()
    summary_rows.append(row)

results_df = pd.DataFrame(summary_rows)

for col, default in [('simulation', ''), ('ok', False), ('n_parcelles', 0), ('reason', '')]:
    if col not in results_df.columns:
        results_df[col] = default

print(f'Resultats : {int(results_df["ok"].sum())} simulations OK')
mean_cols = [c for c in results_df.columns if c.endswith('_mean')]
display_cols = ['simulation', 'ok', 'n_parcelles'] + mean_cols
if 'reason' in results_df.columns and (results_df['reason'].astype(str) != '').any():
    display_cols.append('reason')

display(results_df[display_cols].round(3))

if not results_df['ok'].any():
    print(
        "Aucune sortie exploitable n'a été collectée. Vérifie que les dossiers de logs pointés par `results` "
        "contiennent bien `sorties_CN.csv` avec au moins un couvert cultivé différent de `solnu`."
    )

print('\nDistribution (zone, sol_type, zone_meteo) sur smt_000 :')
_s0 = all_data.get('smt_000', pd.DataFrame())
if not _s0.empty:
    _avg = _s0.groupby(['zone', 'sol_type', 'zone_meteo']).size().reset_index(name='n_parcelles')
    display(_avg)

# --- Sauvegarde detail par simulation ---
results_df.to_csv(XML_DIR / 'resultats_smt.csv', index=False)
for nom_simu, df in all_data.items():
    df.to_csv(XML_DIR / f'detail_{nom_simu}.csv', index=False)
print(f'\nFichiers sauvegardes dans {XML_DIR}/')


In [ ]:
# --- 8. Export for Metamodeling ---
import pandas as pd
import numpy as np

print(f"Collecte des résultats et fusion avec le plan d'expérience ({xt.shape[0]} points)...")

_records = []
parc_to_idx = {p: k for k, p in enumerate(ALL_PARCELLES)}

for nom_simu, out_dir in results.items():
    if not (out_dir and out_dir.exists()):
        continue
    sim_idx = int(nom_simu.split('_')[-1])

    df = read_simulation_outputs(out_dir)
    if df.empty:
        continue

    df['point_idx'] = sim_idx * len(ALL_PARCELLES) + df['parcelle'].map(parc_to_idx)

    df_parc = df.groupby(['parcelle', 'point_idx', 'zone', 'sol_type']).agg({
        'N_lixi': 'mean', 'dCorg': 'mean', 'rdt': 'mean'
    }).reset_index()

    _records.append(df_parc)

if _records:
    df_full = pd.concat(_records, ignore_index=True)

    point_indices = df_full['point_idx'].values
    xt_subset = xt[point_indices]

    for j in range(xt.shape[1]):
        df_full[f'feat_{j}'] = xt_subset[:, j]

    zone_map = {z: i for i, z in enumerate(sorted(df_full['zone'].unique()))}
    soil_map = {s: i for i, s in enumerate(sorted(df_full['sol_type'].unique()))}

    df_full['Milieu (Climat)'] = df_full['zone'].map(zone_map)
    df_full['Milieu (Sol)'] = df_full['sol_type'].map(soil_map)

    X_agri_names = [f'feat_{j}' for j in range(xt.shape[1])]
    export_cols = X_agri_names + ['Milieu (Climat)', 'Milieu (Sol)', 'N_lixi', 'dCorg', 'rdt', 'point_idx', 'parcelle']

    df_export = df_full[export_cols]
    export_path = XML_DIR / 'dataset_metamodel.csv'
    df_export.to_csv(export_path, index=False)

    feature_manifest = pd.DataFrame({
        'colonne': X_agri_names,
        'parametre': SMT_FEATURES[:len(X_agri_names)],
    })
    feature_manifest.to_csv(XML_DIR / 'dataset_metamodel_features.csv', index=False)

    export_path_for_api = PROJECT_LOG_DIR / 'dataset_metamodel.csv'
    df_export.to_csv(export_path_for_api, index=False)
    feature_manifest.to_csv(PROJECT_LOG_DIR / 'dataset_metamodel_features.csv', index=False)

    print(f"Export réussi : {export_path}")
    print(f"Copie API : {export_path_for_api}")
    print(f"Nombre de points : {len(df_export)}")
    print(f"Structure : {len(X_agri_names)} agri + 2 env + 3 sorties")
else:
    print("Aucune donnée collectée.")
